# Melanoma klasifikacija - Kompletni pipeline

Ovaj notebook sadrzi celokupan pipeline za klasifikaciju melanoma:

| Sekcija | Opis | Trajanje |
|---------|------|----------|
| 0. Setup | Instalacija, Drive, src/ | ~2 min |
| 1. Dataset | Kaggle download + statistika | ~15 min |
| 2. Preprocessing | Hair removal, Hu momenti, augmentacija | ~2 min |
| 3. CNN treniranje | Custom CNN, 5-fold CV | ~15-30 min |
| 4. EfficientNet treniranje | Transfer learning, 5-fold CV | ~30-60 min |
| 5. Evaluacija | Poredjenje modela, ROC, metrike | ~1 min |
| 6. Fairness | Analiza pravicnosti po polu i starosti | ~1 min |

**Autori:** Jelena Adamovic, Milos Bojanic  
**Predmet:** SIAP  
**Dataset:** [ISIC Challenge 2020](https://www.kaggle.com/datasets/sumaiyabinteshahid/isic-challenge-dataset-2020)

---
## 0. Setup okruzenja

Proveravamo GPU, instaliramo zavisnosti i povezujemo Google Drive.

### 0.1 Provera GPU-a

**VAZNO:** Pre pokretanja ukljucite GPU: `Runtime -> Change runtime type -> GPU (T4)`

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA dostupna: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memorija: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("UPOZORENJE: GPU NIJE DOSTUPAN!")
    print("Idite na Runtime -> Change runtime type -> GPU")

### 0.2 Instalacija zavisnosti

In [ ]:
!pip install -q timm albumentations mahotas

### 0.3 Google Drive + src/ moduli

Povezujemo Google Drive gde se nalaze nasi `src/` Python moduli i gde cemo sacuvati rezultate.

In [ ]:
import os, sys, shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Kopiraj src/ sa Drive-a na lokalni disk (brze citanje)
REPO_DIR = '/content/melanoma-classification'
SRC_DRIVE = '/content/drive/MyDrive/melanoma_colab/src'
SRC_LOCAL = f'{REPO_DIR}/src'

if not os.path.exists(SRC_LOCAL):
    os.makedirs(REPO_DIR, exist_ok=True)
    shutil.copytree(SRC_DRIVE, SRC_LOCAL)
    print(f"src/ kopiran sa Drive-a na {SRC_LOCAL}")
else:
    print(f"src/ vec postoji na {SRC_LOCAL}")

sys.path.insert(0, REPO_DIR)

# Kreiraj folder za rezultate na Drive-u
RESULTS_DIR = '/content/drive/MyDrive/melanoma_results'
MODELS_DIR = f'{RESULTS_DIR}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Verifikacija
print(f"\nsrc/ moduli dostupni: {os.path.exists(f'{SRC_LOCAL}/config.py')}")
print(f"Rezultati ce se cuvati u: {RESULTS_DIR}")

---
## 1. Preuzimanje dataseta sa Kaggle-a

Dataset: ISIC Challenge 2020 (~23GB, ~33,000 dermoskopskih slika).

**Pre pokretanja:** Potreban vam je Kaggle API kljuc (`kaggle.json`).
Ako nemate, napravite ga na https://www.kaggle.com/settings -> API -> Create New Token.

### 1.1 Kaggle API Setup

In [ ]:
from google.colab import files

if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    print("Upload-ujte vas kaggle.json fajl:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API konfigurisan!")
else:
    print("Kaggle API vec konfigurisan.")

### 1.2 Preuzimanje i raspakivanje

Ovo traje ~10-15 minuta. **Nemojte zatvarati tab dok traje!**

In [ ]:
DATA_DIR = '/content/data'

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    print("Preuzimanje dataseta sa Kaggle-a (~23GB)...")
    !kaggle datasets download -d sumaiyabinteshahid/isic-challenge-dataset-2020 -p {DATA_DIR}
    print("\nRaspakivanje...")
    !unzip -q {DATA_DIR}/*.zip -d {DATA_DIR}
    !rm -f {DATA_DIR}/*.zip
    print("Gotovo!")
else:
    print(f"Dataset vec postoji na {DATA_DIR}")

### 1.3 Pronalazenje strukture dataseta

In [ ]:
import glob
import pandas as pd

# Pronadji CSV i slike
csv_files = glob.glob(f"{DATA_DIR}/**/train.csv", recursive=True)
jpg_files = glob.glob(f"{DATA_DIR}/**/*.jpg", recursive=True)

TRAIN_CSV = csv_files[0] if csv_files else None
TRAIN_DIR = os.path.dirname(jpg_files[0]) if jpg_files else None

print(f"TRAIN_CSV = {TRAIN_CSV}")
print(f"TRAIN_DIR = {TRAIN_DIR}")
print(f"Ukupno JPG slika: {len(jpg_files)}")

# Statistika dataseta
df_info = pd.read_csv(TRAIN_CSV)
print(f"\nBroj redova u CSV: {len(df_info)}")
print(f"Kolone: {list(df_info.columns)}")
print(f"\nRaspodela klasa:")
print(f"  Benign:    {(df_info['target'] == 0).sum()} ({(df_info['target'] == 0).mean()*100:.1f}%)")
print(f"  Malignant: {(df_info['target'] == 1).sum()} ({(df_info['target'] == 1).mean()*100:.1f}%)")
print(f"\nBroj pacijenata: {df_info['patient_id'].nunique()}")

**Zakljucak:** Dataset je jako neubalansiran (~98% benignih, ~2% malignih). Koristicemo `pos_weight` u loss funkciji i stratifikovanu cross-validaciju.

### 1.4 Primer slike iz dataseta

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample = df_info.iloc[0]
sample_path = os.path.join(TRAIN_DIR, f"{sample['image_name']}.jpg")
img = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f"{sample['image_name']} - target: {sample['target']}")
plt.axis('off')
plt.show()
print(f"Dimenzije slike: {img.shape}")

---
## 2. Konfiguracija i preprocessing

Postavljamo konfiguraciju i prikazujemo korake preprocesiranja.

### 2.1 Konfiguracija

In [ ]:
from src.config import Config

cfg = Config.colab()
cfg.train_csv = TRAIN_CSV
cfg.train_dir = TRAIN_DIR
cfg.model_save_dir = MODELS_DIR

print(f"Device: {cfg.get_device()}")
print(f"Epochs: {cfg.epochs}")
print(f"Folds: {cfg.num_folds}")
print(f"Batch size: {cfg.batch_size}")
print(f"Train CSV: {cfg.train_csv}")
print(f"Train DIR: {cfg.train_dir}")
print(f"Models DIR: {cfg.model_save_dir}")

### 2.2 Uklanjanje dlaka (Hair Removal)

Dermoskopske slike cesto sadrze dlake koje mogu omesti klasifikaciju. Koristimo **Black-Hat morfolosku transformaciju** za detekciju dlaka, a zatim **Telea inpainting algoritam** za njihovo uklanjanje.

Postupak:
1. Konvertovanje u grayscale
2. Black-Hat transformacija sa strukturnim elementom (15x15)
3. Binarizacija maske (prag = 10)
4. Inpainting koristeci Telea algoritam

In [ ]:
import numpy as np
from src.preprocessing import remove_hairs
from src.visualization import plot_preprocessing_examples

fig = plot_preprocessing_examples(cfg.train_dir, n=4)
plt.show()

**Zakljucak:** Black-Hat transformacija uspesno detektuje tamne linearne strukture (dlake) na svetloj kozi, a Telea inpainting ih zamenjuje okolnim pikselima.

### 2.3 Ekstrakcija Hu momenata

**Hu momenti** su 7 invarijantnih momenata koji opisuju oblik objekta na slici. Invarijantni su na translaciju, skaliranje i rotaciju. Koristimo **prva 4 Hu momenta** sa log-transformacijom za numericku stabilnost.

In [ ]:
from src.features import calculate_hu_moments

img_files = [f for f in os.listdir(cfg.train_dir) if f.endswith('.jpg')][:8]
hu_data = []

for fname in img_files:
    img = cv2.imread(os.path.join(cfg.train_dir, fname))
    cleaned = remove_hairs(img)
    hu = calculate_hu_moments(cleaned, n_moments=4)
    hu_data.append({
        'image': fname[:20],
        'Hu1': f'{hu[0]:.4f}',
        'Hu2': f'{hu[1]:.4f}',
        'Hu3': f'{hu[2]:.4f}',
        'Hu4': f'{hu[3]:.4f}',
    })

hu_df = pd.DataFrame(hu_data)
print("Hu momenti za primere slika (log-transformisani):")
hu_df

**Zakljucak:** Hu momenti se razlikuju izmedju slika, sto znaci da nose informaciju o obliku lezije. Koriste se kao dodatni ulaz u model (late fusion).

### 2.4 Enkodiranje metapodataka

Pored vizuelnih obelezja, koristimo i metapodatke pacijenata:
- **Pol** (sex) - one-hot enkodiranje
- **Anatomska lokacija** (anatom_site_general_challenge) - one-hot enkodiranje
- **Starost** (age_approx) - min-max normalizacija na [0, 1]

In [ ]:
from src.features import encode_metadata

df_demo = pd.read_csv(cfg.train_csv)
df_encoded, metadata_cols, metadata_dim = encode_metadata(df_demo)

print(f"Metadata kolone ({metadata_dim}): {metadata_cols}")
print(f"\nPrimer enkodiranih metapodataka (prvih 5 redova):")
df_encoded[metadata_cols].head()

**Zakljucak:** Svaki pacijent ima numericki vektor metapodataka. Ovaj vektor se spaja sa Hu momentima (4 vrednosti) u drugoj grani modela (late fusion).

### 2.5 Augmentacija podataka

Zbog izrazitog disbalansa klasa koristimo **on-the-fly augmentaciju** tokom treniranja:
- Horizontalni i vertikalni flip, rotacija za 90 stepeni
- Shift-Scale-Rotate, ColorJitter
- Normalizacija (ImageNet mean/std)

Augmentacija se primenjuje samo na trening skup. Ovaj pristup ne moze da izazove curenje podataka izmedju foldova.

In [ ]:
from src.augmentation import get_train_transforms

train_t = get_train_transforms(cfg.image_size)

sample_img = cv2.imread(os.path.join(cfg.train_dir, img_files[0]))
sample_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(sample_rgb)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i in range(1, 8):
    row, col = divmod(i, 4)
    augmented = train_t(image=sample_rgb)['image']
    img_show = augmented.permute(1, 2, 0).numpy()
    img_show = img_show * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_show = np.clip(img_show, 0, 1)
    axes[row, col].imshow(img_show)
    axes[row, col].set_title(f'Augmentacija {i}')
    axes[row, col].axis('off')

fig.suptitle('Primeri augmentacije tokom treniranja', fontsize=13)
plt.tight_layout()
plt.show()

**Zakljucak:** Augmentacija generise razlicite varijacije iste slike, sto smanjuje overfitting, posebno za malobrojnu malignu klasu.

### 2.6 Offline preprocesiranje svih slika (JEDNOM)

Ovo je kljucni korak za performanse! Umesto da za svaku sliku radimo resize + hair removal + Hu momente tokom treniranja (sto se ponavlja svaku epohu), **uradicemo to jednom za sve slike** i sacuvati rezultate.

Preprocesiramo za oba modela unapred:
- **128x128** za Custom CNN
- **224x224** za EfficientNet

Ovo traje ~20-30 minuta, ali posle toga treniranje je **drasticno brze** jer DataLoader samo cita gotove numpy fajlove.

In [ ]:
from src.preprocessing_cache import preprocess_and_cache
from src.data_utils import load_and_prepare_data

# Ucitaj podatke (potrebno za listu slika)
df = load_and_prepare_data(cfg)
print(f"Ukupno slika za preprocesiranje: {len(df)}")

# Preprocesiraj za CNN (128x128)
CNN_CACHE = "/content/preprocessed_128"
print("
[1/2] Preprocesiranje za CNN (128x128)...")
preprocess_and_cache(df, cfg.train_dir, CNN_CACHE, image_size=128,
                     apply_hair_removal=cfg.apply_hair_removal)

# Preprocesiraj za EfficientNet (224x224)
EFFNET_CACHE = "/content/preprocessed_224"
print("
[2/2] Preprocesiranje za EfficientNet (224x224)...")
preprocess_and_cache(df, cfg.train_dir, EFFNET_CACHE, image_size=224,
                     apply_hair_removal=cfg.apply_hair_removal)

print("
Preprocesiranje zavrseno! Slike su spremne za treniranje.")

**Zakljucak:** Sve slike su preprocesirane i sacuvane kao numpy fajlovi. Tokom treniranja, DataLoader samo cita gotove fajlove umesto da ponovo radi resize, hair removal i ekstrakciju Hu momenata za svaku sliku svaku epohu. Ovo ubrzava treniranje **10-100x**.

---
## 3. Treniranje Custom CNN modela

Nas Custom CNN model se sastoji od:
- **3 konvoluciona sloja** (32->64->128 filtera) sa BatchNorm i ReLU
- **AdaptiveAvgPool** -> vektor od 128 dimenzija
- **Late fusion**: spajanje vizuelnih obelezja (128-dim) sa metapodacima (Hu momenti + encoded metadata)
- **Klasifikator**: FC(128+feat_dim -> 64) -> FC(64 -> 1)

Treniramo sa:
- **StratifiedGroupKFold** - podela po pacijentima (sprecava curenje podataka)
- **BCEWithLogitsLoss** sa pos_weight za balansiranje klasa
- **Early stopping** na osnovu validacionog AUC-ROC

In [ ]:
from src.data_utils import load_and_prepare_data
from src.training import run_cross_validation
from src.visualization import plot_all_folds_losses, plot_roc_curve, plot_fold_comparison

### 3.1 Konfiguracija CNN-a

In [ ]:
cfg.model_type = 'cnn'
cfg.image_size = 128
cfg.cache_dir = CNN_CACHE  # Koristi preprocesirane slike

print(f"Model: {cfg.model_type}")
print(f"Image size: {cfg.image_size}x{cfg.image_size}")
print(f"Epochs: {cfg.epochs}, Folds: {cfg.num_folds}")
print(f"Batch size: {cfg.batch_size}")
print(f"Device: {cfg.get_device()}")

### 3.2 Ucitavanje podataka

In [ ]:
df = load_and_prepare_data(cfg)

print(f"Ukupno slika: {len(df)}")
print(f"Raspodela klasa:")
print(f"  Benign:    {(df['target'] == 0).sum()} ({(df['target'] == 0).mean()*100:.1f}%)")
print(f"  Malignant: {(df['target'] == 1).sum()} ({(df['target'] == 1).mean()*100:.1f}%)")
print(f"Broj pacijenata: {df['patient_id'].nunique()}")
print(f"Feature dim: {cfg.feature_dim}")

### 3.3 Treniranje sa k-fold cross-validacijom

Sa GPU-om, ovo traje ~15-30 minuta.

In [ ]:
cnn_results = run_cross_validation(df, cfg)

### 3.4 Vizualizacija rezultata CNN-a

In [ ]:
# Krive gubitka po foldovima
fig = plot_all_folds_losses(cnn_results['per_fold_results'])
plt.show()

**Interpretacija:** Train loss opada kroz epohe. Ako val loss pocne da raste dok train loss opada, to ukazuje na overfitting â€” early stopping zaustavlja treniranje pre toga.

In [ ]:
# ROC kriva
fig = plot_roc_curve(
    cnn_results['oof_labels'],
    cnn_results['oof_probs'],
    fold_results=cnn_results['per_fold_results'],
    label='Custom CNN'
)
plt.show()

In [ ]:
# Poredjenje AUC po foldovima
fig = plot_fold_comparison(cnn_results['per_fold_results'])
plt.show()

print(f"\nCNN Mean AUC: {cnn_results['mean_auc']:.4f} +/- {cnn_results['std_auc']:.4f}")

### 3.5 Cuvanje CNN rezultata na Drive

In [ ]:
import pickle

cnn_save_path = os.path.join(RESULTS_DIR, 'cnn_results.pkl')
with open(cnn_save_path, 'wb') as f:
    pickle.dump({
        'oof_labels': cnn_results['oof_labels'],
        'oof_probs': cnn_results['oof_probs'],
        'oof_df': cnn_results['oof_df'],
        'mean_auc': cnn_results['mean_auc'],
        'std_auc': cnn_results['std_auc'],
        'per_fold_aucs': [r['best_val_auc'] for r in cnn_results['per_fold_results']],
    }, f)

print(f"CNN rezultati sacuvani u: {cnn_save_path}")

**Zakljucak:** Custom CNN model je treniran. Rezultati su sacuvani na Google Drive-u za slucaj da se sesija prekine pre EfficientNet treniranja.

---
## 4. Treniranje EfficientNet B0 modela

**EfficientNet B0** je pretreniran model na ImageNet datasetu (transfer learning).

Arhitektura:
- **EfficientNet B0 backbone** (pretreniran na ImageNet) -> 1280-dim vektor
- **Late fusion**: spajanje sa metapodacima (Hu momenti + encoded metadata)
- **Klasifikator**: FC(1280+feat_dim -> 256) -> FC(256 -> 1)

Ocekujemo bolje rezultate od Custom CNN-a jer koristi transferisano znanje sa milionima slika.

### 4.1 Konfiguracija EfficientNet-a

In [ ]:
cfg.model_type = 'efficientnet'
cfg.image_size = 224
cfg.cache_dir = EFFNET_CACHE  # Koristi preprocesirane slike

print(f"Model: {cfg.model_type}")
print(f"Image size: {cfg.image_size}x{cfg.image_size}")
print(f"Epochs: {cfg.epochs}, Folds: {cfg.num_folds}")
print(f"Batch size: {cfg.batch_size}")
print(f"Device: {cfg.get_device()}")

### 4.2 Treniranje sa k-fold cross-validacijom

Sa GPU-om, ovo traje ~30-60 minuta (duze od CNN-a jer su vece slike i slozeniji model).

In [ ]:
effnet_results = run_cross_validation(df, cfg)

### 4.3 Vizualizacija rezultata EfficientNet-a

In [ ]:
fig = plot_all_folds_losses(effnet_results['per_fold_results'])
plt.show()

In [ ]:
fig = plot_roc_curve(
    effnet_results['oof_labels'],
    effnet_results['oof_probs'],
    fold_results=effnet_results['per_fold_results'],
    label='EfficientNet B0'
)
plt.show()

In [ ]:
fig = plot_fold_comparison(effnet_results['per_fold_results'])
plt.show()

print(f"\nEfficientNet Mean AUC: {effnet_results['mean_auc']:.4f} +/- {effnet_results['std_auc']:.4f}")

### 4.4 Cuvanje EfficientNet rezultata na Drive

In [ ]:
effnet_save_path = os.path.join(RESULTS_DIR, 'effnet_results.pkl')
with open(effnet_save_path, 'wb') as f:
    pickle.dump({
        'oof_labels': effnet_results['oof_labels'],
        'oof_probs': effnet_results['oof_probs'],
        'oof_df': effnet_results['oof_df'],
        'mean_auc': effnet_results['mean_auc'],
        'std_auc': effnet_results['std_auc'],
        'per_fold_aucs': [r['best_val_auc'] for r in effnet_results['per_fold_results']],
    }, f)

print(f"EfficientNet rezultati sacuvani u: {effnet_save_path}")

**Zakljucak:** EfficientNet model je treniran. Oba modela su sacuvana na Drive-u.

---
## 5. Evaluacija - Poredjenje modela

Poredimo performanse Custom CNN i EfficientNet B0 modela koristeci out-of-fold (OOF) predikcije iz cross-validacije.

Kljucne metrike:
- **AUC-ROC** â€” primarna metrika, nezavisna od izbora praga klasifikacije. Veci AUC = bolji model.
- **Recall (senzitivnost)** â€” procenat ispravno detektovanih malignih slucajeva. Kriticna metrika u medicinskoj dijagnostici jer ne zelimo da propustimo maligne slucajeve.
- **Precision** â€” procenat tacnih medju pozitivnim predikcijama.

In [ ]:
from src.evaluation import compute_metrics, compute_metrics_at_best_threshold
from src.visualization import plot_roc_comparison, plot_confusion_matrix

### 5.1 Poredjenje ROC krivih

ROC kriva prikazuje odnos izmedju True Positive Rate (Recall) i False Positive Rate za razlicite pragove klasifikacije.

In [ ]:
fig = plot_roc_comparison({
    'Custom CNN': cnn_results,
    'EfficientNet B0': effnet_results,
})
plt.show()

print(f"CNN AUC:         {cnn_results['mean_auc']:.4f} +/- {cnn_results['std_auc']:.4f}")
print(f"EfficientNet AUC: {effnet_results['mean_auc']:.4f} +/- {effnet_results['std_auc']:.4f}")

### 5.2 Detaljne metrike klasifikacije

Izracunavamo metrike za oba modela koristeci:
1. **Fiksni prag** (0.5) â€” standardni default
2. **Optimalni prag** â€” maksimizuje Youden's J statistiku (TPR - FPR)

In [ ]:
cnn_metrics = compute_metrics(cnn_results['oof_labels'], cnn_results['oof_probs'], threshold=0.5)
effnet_metrics = compute_metrics(effnet_results['oof_labels'], effnet_results['oof_probs'], threshold=0.5)

cnn_optimal, cnn_thresh = compute_metrics_at_best_threshold(cnn_results['oof_labels'], cnn_results['oof_probs'])
effnet_optimal, effnet_thresh = compute_metrics_at_best_threshold(effnet_results['oof_labels'], effnet_results['oof_probs'])

comparison = pd.DataFrame({
    'Metrika': ['AUC-ROC', 'Accuracy', 'Recall', 'Precision', 'F1-score', 'Prag'],
    'CNN (prag=0.5)': [
        f"{cnn_metrics['auc_roc']:.4f}",
        f"{cnn_metrics['accuracy']:.4f}",
        f"{cnn_metrics['recall']:.4f}",
        f"{cnn_metrics['precision']:.4f}",
        f"{cnn_metrics['f1']:.4f}",
        '0.5000',
    ],
    'CNN (optimalni)': [
        f"{cnn_optimal['auc_roc']:.4f}",
        f"{cnn_optimal['accuracy']:.4f}",
        f"{cnn_optimal['recall']:.4f}",
        f"{cnn_optimal['precision']:.4f}",
        f"{cnn_optimal['f1']:.4f}",
        f"{cnn_thresh:.4f}",
    ],
    'EfficientNet (prag=0.5)': [
        f"{effnet_metrics['auc_roc']:.4f}",
        f"{effnet_metrics['accuracy']:.4f}",
        f"{effnet_metrics['recall']:.4f}",
        f"{effnet_metrics['precision']:.4f}",
        f"{effnet_metrics['f1']:.4f}",
        '0.5000',
    ],
    'EfficientNet (optimalni)': [
        f"{effnet_optimal['auc_roc']:.4f}",
        f"{effnet_optimal['accuracy']:.4f}",
        f"{effnet_optimal['recall']:.4f}",
        f"{effnet_optimal['precision']:.4f}",
        f"{effnet_optimal['f1']:.4f}",
        f"{effnet_thresh:.4f}",
    ],
})

print("Poredjenje modela (Out-of-Fold predikcije):")
comparison

**Napomena o Recall-u:** U medicinskoj dijagnostici, Recall (senzitivnost) je kriticna metrika jer minimizuje **lazno negativne** rezultate â€” ne zelimo da propustimo maligne slucajeve. Visok Recall sa razumnim Precision je cilj.

### 5.3 Matrice konfuzije

Prikazujemo matrice konfuzije za oba modela sa **optimalnim pragom**.

In [ ]:
print("=== Custom CNN ===")
cnn_preds_opt = (cnn_results['oof_probs'] >= cnn_thresh).astype(int)
fig = plot_confusion_matrix(cnn_results['oof_labels'], cnn_preds_opt,
                            title=f'Custom CNN (prag={cnn_thresh:.3f})')
plt.show()

print("\n=== EfficientNet B0 ===")
effnet_preds_opt = (effnet_results['oof_probs'] >= effnet_thresh).astype(int)
fig = plot_confusion_matrix(effnet_results['oof_labels'], effnet_preds_opt,
                            title=f'EfficientNet B0 (prag={effnet_thresh:.3f})')
plt.show()

### 5.4 Classification Report

In [ ]:
print("=== Custom CNN (optimalni prag) ===")
print(cnn_optimal['classification_report'])

print("\n=== EfficientNet B0 (optimalni prag) ===")
print(effnet_optimal['classification_report'])

**Zakljucak:** EfficientNet B0 sa transfer learning-om ocekivano daje bolje rezultate od Custom CNN-a. Kljucna prednost je veci AUC-ROC i bolji Recall za malignu klasu.

---
## 6. Analiza pravicnosti (Fairness)

Analiziramo pravicnost modela koristeci **Equalized Odds** metriku.

**Equalized Odds** zahteva da model ima jednake performanse (TPR i FPR) za sve demografske grupe:
- **TPR (True Positive Rate / Recall)** â€” procenat ispravno detektovanih malignih slucajeva
- **FPR (False Positive Rate)** â€” procenat pogresno klasifikovanih benignih slucajeva

Analiziramo pravicnost po:
1. **Polu** (muski/zenski)
2. **Starosnoj grupi** (<30, 30-44, 45-59, 60-74, 75+)

In [ ]:
from src.fairness import full_fairness_report
from src.visualization import plot_fairness_bars

# Koristimo model sa boljim AUC
if effnet_results['mean_auc'] >= cnn_results['mean_auc']:
    best_results = effnet_results
    best_model_name = 'EfficientNet B0'
else:
    best_results = cnn_results
    best_model_name = 'Custom CNN'

oof_labels = best_results['oof_labels']
oof_probs = best_results['oof_probs']
oof_df = best_results['oof_df']

_, best_threshold = compute_metrics_at_best_threshold(oof_labels, oof_probs)

print(f"Model za analizu: {best_model_name}")
print(f"AUC: {best_results['mean_auc']:.4f}")
print(f"Optimalni prag: {best_threshold:.4f}")
print(f"Broj uzoraka: {len(oof_labels)}")

In [ ]:
report = full_fairness_report(oof_df, oof_labels, oof_probs, threshold=best_threshold)

### 6.1 Pravicnost po polu

Analiziramo da li model jednako dobro prepoznaje melanom kod muskih i zenskih pacijenata.

In [ ]:
if 'sex' in report:
    print("Equalized Odds po polu:")
    display(report['sex'])

    tpr_disp = report['sex'].attrs.get('tpr_disparity', float('nan'))
    fpr_disp = report['sex'].attrs.get('fpr_disparity', float('nan'))
    print(f"\nTPR disparitet: {tpr_disp:.4f}" if not np.isnan(tpr_disp) else "\nTPR disparitet: N/A")
    print(f"FPR disparitet: {fpr_disp:.4f}" if not np.isnan(fpr_disp) else "FPR disparitet: N/A")

    fig = plot_fairness_bars(report['sex'], title=f'Equalized Odds po polu ({best_model_name})')
    plt.show()
else:
    print("Kolona 'sex' nije dostupna u podacima.")

**Interpretacija:** Idealno, TPR i FPR bi trebalo da budu jednaki za obe grupe. Veliki disparitet ukazuje na to da model moze biti pristrastan prema jednom polu.

### 6.2 Pravicnost po starosnoj grupi

Melanom je cesci kod starijih pacijenata. Proveravamo da li model jednako dobro funkcionise za sve starosne grupe.

In [ ]:
if 'age_group' in report:
    print("Equalized Odds po starosnoj grupi:")
    display(report['age_group'])

    tpr_disp = report['age_group'].attrs.get('tpr_disparity', float('nan'))
    fpr_disp = report['age_group'].attrs.get('fpr_disparity', float('nan'))
    print(f"\nTPR disparitet: {tpr_disp:.4f}" if not np.isnan(tpr_disp) else "\nTPR disparitet: N/A")
    print(f"FPR disparitet: {fpr_disp:.4f}" if not np.isnan(fpr_disp) else "FPR disparitet: N/A")

    fig = plot_fairness_bars(report['age_group'], title=f'Equalized Odds po starosti ({best_model_name})')
    plt.show()
else:
    print("Kolona 'age_approx' nije dostupna u podacima.")

### 6.3 Sumarni pregled dispariteta

In [ ]:
print(f"=== Sumarni pregled dispariteta za {best_model_name} ===")
print(f"{'Grupa':<20} {'TPR disparitet':>15} {'FPR disparitet':>15}")
print('-' * 52)

for group_name, fair_df in report.items():
    tpr_d = fair_df.attrs.get('tpr_disparity', float('nan'))
    fpr_d = fair_df.attrs.get('fpr_disparity', float('nan'))
    tpr_str = f"{tpr_d:.4f}" if not np.isnan(tpr_d) else "N/A"
    fpr_str = f"{fpr_d:.4f}" if not np.isnan(fpr_d) else "N/A"
    print(f"{group_name:<20} {tpr_str:>15} {fpr_str:>15}")

print()
print("Manji disparitet = pravicniji model (idealno = 0.0)")

**Zakljucak fairness analize:**

- Disparitet u TPR ukazuje na razliku u sposobnosti detekcije melanoma izmedju grupa
- Disparitet u FPR ukazuje na razliku u broju lazno pozitivnih rezultata
- Za medicinsku primenu, posebno je vazno da TPR bude visok i ujednacen za SVE grupe

**Moguca poboljsanja:**
- Veci i raznovrsniji dataset (posebno za nedovoljno zastupljene grupe)
- Augmentacija podataka fokusirana na manje zastupljene grupe
- Fairness-aware tehnike treniranja (npr. adversarial debiasing)
- Kalibracija modela po grupama

---
## 7. Rezime rezultata

Kompletni pipeline za klasifikaciju melanoma je zavrsen.

In [ ]:
print("=" * 60)
print("  REZIME REZULTATA")
print("=" * 60)
print(f"\nDataset: {len(df)} slika, {df['patient_id'].nunique()} pacijenata")
print(f"Cross-validacija: {cfg.num_folds}-fold StratifiedGroupKFold")
print(f"\n{'Model':<20} {'Mean AUC':>12} {'Std':>10}")
print('-' * 44)
print(f"{'Custom CNN':<20} {cnn_results['mean_auc']:>12.4f} {cnn_results['std_auc']:>10.4f}")
print(f"{'EfficientNet B0':<20} {effnet_results['mean_auc']:>12.4f} {effnet_results['std_auc']:>10.4f}")
print(f"\nBolji model: {best_model_name}")
print(f"\nSacuvani fajlovi na Google Drive-u ({RESULTS_DIR}):")
print(f"  - cnn_results.pkl")
print(f"  - effnet_results.pkl")
print(f"  - models/ (sacuvani modeli po foldovima)")
print("\nGotovo!")